# Worker del analizador de chats en Google Colab

1. (Opcional) *Entorno de ejecución → Cambiar tipo → GPU T4*.
2. *Entorno de ejecución → Ejecutar todas*.
3. Cuando lo pida, pegá la **URL de CloudAMQP** (`amqps://…`) y el **WORKER_TOKEN** de `distribuido/config.env`.

La última celda se queda corriendo: ese es el worker. Sin túnel a la laptop, atiende **solo texto** (las fotos y los audios los procesa el worker de la laptop).

⚠️ Procesar con estos workers **solo el dataset generado**, nunca chats reales.

In [ ]:
# 1. Código del proyecto
import os, subprocess
REPO = "https://github.com/SharonBlanco/Analizador-de-Chats-de-Whatsapp.git"
if not os.path.isdir("/content/proyecto"):
    !git clone -q {REPO} /content/proyecto
else:
    !git -C /content/proyecto pull -q
%cd /content/proyecto
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "Sin GPU: el worker usará la CPU" 

In [ ]:
# 2. Librerías (Colab ya trae PyTorch con CUDA y ffmpeg)
!pip install -q pika==1.3.2 pysentimiento ultralytics faster-whisper

In [ ]:
# 3. Configuración: se pide al correr (no queda escrita en el notebook)
from getpass import getpass
def dato(clave, pregunta, opcional=False):
    try:                                   # si existe en los Secretos de Colab (ícono de llave), se usa
        from google.colab import userdata
        v = userdata.get(clave)
        if v:
            return v
    except Exception:
        pass
    return getpass(pregunta) if not opcional else input(pregunta).strip()

os.environ["RABBIT_URL"] = dato("RABBIT_URL", "URL de CloudAMQP (amqps://…): ")
os.environ["WORKER_TOKEN"] = dato("WORKER_TOKEN", "WORKER_TOKEN: ")
tunel = dato("COORDINADOR_URL", "Túnel a la laptop (https://…trycloudflare.com) o Enter si no hay: ", opcional=True)

NOMBRE = "colab-1"                     # cambiar a colab-2 en el segundo worker
if tunel:
    os.environ["COORDINADOR_URL"] = tunel
    POOLS = "texto,imagen,audio,adjuntos"
else:
    POOLS = "texto"                    # sin túnel no puede descargar fotos ni audios de la laptop
os.environ.update(WORKER_ID=NOMBRE, HOST_NAME=NOMBRE, WORKER_POOLS=POOLS, PYTHONUNBUFFERED="1")
print("Worker", NOMBRE, "· pools:", POOLS)

In [ ]:
# 4. Worker (se queda corriendo). El supervisor lo reinicia si se queda sin memoria (M1).
!python distribuido/worker/supervisor.py